# Expanded Multi-Source Cultural Salience Composite (v2)

Incorporate the three new sources from notebook 07 into the composite score:

| Signal Column | Sources | What It Measures |
|---|---|---|
| `pctile_jeopardy` | Jeopardy (existing) | Quiz-show salience |
| `pctile_wiki` | Wikipedia pageviews (existing) | Public attention |
| `pctile_xw` | Crossword frequency (existing) | Puzzle-constructor assumptions |
| `pctile_proverb` | Wiktextract + Bartlett's | Canonical sayings/phrases |
| `pctile_notable` | Wikidata sitelink count (figures) | Global cultural figure recognition |
| `pctile_geo` | Wikidata sitelink count (places) | Geographic literacy |

Key architectural change: **full-outer-join** — new sources can introduce concepts with zero Jeopardy signal.

In [ ]:
import pandas as pd
import numpy as np
import re
from scipy import stats
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

DATA_DIR = "../data"

def normalize(text):
    """Normalize a term for matching."""
    if not isinstance(text, str):
        return ""
    s = text.lower().strip()
    s = re.sub(r"^(the|a|an)\s+", "", s)
    s = re.sub(r"[,;:!?'\"()\[\]]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

print("Ready.")

## 1. Load Existing Composite + New Sources

In [ ]:
# Load existing composite (v1)
composite_v1 = pd.read_csv(f"{DATA_DIR}/cultural_salience_composite.csv")
print(f"Existing composite v1: {len(composite_v1):,} concepts")
print(f"  Columns: {list(composite_v1.columns)}")

# Add norm column for matching
composite_v1["norm"] = composite_v1["concept"].apply(normalize)
composite_v1["norm_display"] = composite_v1["display_name"].apply(normalize)

# Load new sources
proverbs = pd.read_csv(f"{DATA_DIR}/wiktextract_proverbs_idioms.csv")
quotations = pd.read_csv(f"{DATA_DIR}/bartletts_quotations.csv")
figures = pd.read_csv(f"{DATA_DIR}/wikidata_global_figures.csv")
places = pd.read_csv(f"{DATA_DIR}/wikidata_notable_places.csv")

print(f"\nNew sources:")
print(f"  Proverbs/idioms: {len(proverbs):,}")
print(f"  Bartlett's quotations: {len(quotations):,}")
print(f"  Wikidata figures: {len(figures):,}")
print(f"  Wikidata places: {len(places):,}")

## 2. Merge Proverb/Idiom Signal

Combine Wiktextract proverbs/idioms and Bartlett's quotations into a single "proverb" signal.
Signal value: 2 if in both sources, 1 if in one source.

In [ ]:
# Build a proverb/quotation lookup: norm -> signal strength
proverb_norms = set(proverbs["norm"].dropna())
quotation_norms = set(quotations["norm"].dropna())

# Check existing concepts for proverb matches
existing_norms = set(composite_v1["norm"].dropna())
existing_display_norms = set(composite_v1["norm_display"].dropna())
all_existing = existing_norms | existing_display_norms

proverbs_in_existing = proverb_norms & all_existing
quotes_in_existing = quotation_norms & all_existing

print(f"Proverbs matching existing concepts: {len(proverbs_in_existing):,}")
print(f"Quotations matching existing concepts: {len(quotes_in_existing):,}")

# New concepts from proverbs/quotations (not in existing composite)
new_proverbs = proverb_norms - all_existing
new_quotes = quotation_norms - all_existing
print(f"\nNew proverbs/idioms (not in existing): {len(new_proverbs):,}")
print(f"New quotations (not in existing): {len(new_quotes):,}")

In [ ]:
# Assign proverb signal to existing concepts
def proverb_signal(row):
    """Compute proverb signal: 2 if in both sources, 1 if in one."""
    n = row["norm"]
    nd = row.get("norm_display", "")
    score = 0
    if n in proverb_norms or nd in proverb_norms:
        score += 1
    if n in quotation_norms or nd in quotation_norms:
        score += 1
    return score if score > 0 else np.nan

composite_v1["proverb_signal"] = composite_v1.apply(proverb_signal, axis=1)
print(f"Existing concepts with proverb signal: {composite_v1['proverb_signal'].notna().sum():,}")
print(composite_v1["proverb_signal"].value_counts().sort_index().to_string())

## 3. Merge Notable Figures Signal

For existing PERSON concepts: add sitelink count as `notable_signal`.
For new figures: create new rows.

In [ ]:
# Match figures to existing concepts by QID first, then by normalized name
# Load wikidata_linking for QID matching
wikidata_linking = pd.read_csv(f"{DATA_DIR}/wikidata_linking.csv")
concept_to_qid = dict(zip(wikidata_linking["concept"], wikidata_linking["qid"]))

# Map existing concepts to their QID
composite_v1["qid"] = composite_v1["concept"].map(concept_to_qid)

# Build figure sitelink lookup by QID and by norm
fig_sitelinks_by_qid = dict(zip(figures["qid"], figures["sitelinks"]))
fig_sitelinks_by_norm = dict(zip(figures["norm"], figures["sitelinks"]))

# Only assign notable_signal to PERSON entity_type concepts (avoid name collisions)
def notable_signal(row):
    if row["entity_type"] != "PERSON":
        return np.nan
    qid = row.get("qid")
    if pd.notna(qid) and qid in fig_sitelinks_by_qid:
        return fig_sitelinks_by_qid[qid]
    n = row["norm"]
    if n in fig_sitelinks_by_norm:
        return fig_sitelinks_by_norm[n]
    return np.nan

composite_v1["notable_signal"] = composite_v1.apply(notable_signal, axis=1)

print(f"Existing PERSON concepts with notable_signal: {composite_v1['notable_signal'].notna().sum():,}")
print(f"\nTop 20 by sitelinks:")
top_notable = composite_v1[composite_v1["notable_signal"].notna()].nlargest(20, "notable_signal")
print(top_notable[["concept", "display_name", "notable_signal", "frequency"]].to_string(index=False))

## 4. Merge Geographic Signal

For existing GPE/LOC concepts: add sitelink count as `geo_signal`.
For new places: create new rows.

In [ ]:
# Build place sitelink lookup by QID and by norm
place_sitelinks_by_qid = dict(zip(places["qid"], places["sitelinks"]))
place_sitelinks_by_norm = dict(zip(places["norm"], places["sitelinks"]))

# Only assign geo_signal to GPE or LOC entity_type (avoid name collisions like Paris)
GEO_TYPES = {"GPE", "LOC", "FAC"}

def geo_signal(row):
    if row["entity_type"] not in GEO_TYPES:
        return np.nan
    qid = row.get("qid")
    if pd.notna(qid) and qid in place_sitelinks_by_qid:
        return place_sitelinks_by_qid[qid]
    n = row["norm"]
    if n in place_sitelinks_by_norm:
        return place_sitelinks_by_norm[n]
    return np.nan

composite_v1["geo_signal"] = composite_v1.apply(geo_signal, axis=1)

print(f"Existing GPE/LOC concepts with geo_signal: {composite_v1['geo_signal'].notna().sum():,}")
print(f"\nTop 20 by sitelinks:")
top_geo = composite_v1[composite_v1["geo_signal"].notna()].nlargest(20, "geo_signal")
print(top_geo[["concept", "display_name", "entity_type", "geo_signal", "frequency"]].to_string(index=False))

## 5. Append New Concepts (Full Outer Join)

Add concepts from new sources that don't match any existing concept.

In [ ]:
# Collect new concepts to append
new_rows = []

# New proverbs/idioms
for _, row in proverbs.iterrows():
    n = row["norm"]
    if n not in all_existing and len(n) >= 3:
        new_rows.append({
            "concept": n,
            "display_name": row["headword"],
            "entity_type": "PROVERB_IDIOM",
            "proverb_signal": 1,
            "source_origin": "wiktextract",
        })

# New quotations
for _, row in quotations.iterrows():
    n = row["norm"]
    if n not in all_existing and len(n) >= 3:
        # Check if already added as a proverb
        existing_new_norms = {r["concept"] for r in new_rows}
        if n in existing_new_norms:
            # Upgrade signal to 2 (in both sources)
            for r in new_rows:
                if r["concept"] == n:
                    r["proverb_signal"] = 2
                    break
        else:
            new_rows.append({
                "concept": n,
                "display_name": row["quotation"],
                "entity_type": "QUOTATION",
                "proverb_signal": 1,
                "source_origin": "bartletts",
            })

# New figures (not already in Jeopardy)
figures_new = figures[~figures["already_in_jeopardy"]].copy()
# Also check norm against existing
figures_new = figures_new[~figures_new["norm"].isin(all_existing)]
existing_new_norms = {r["concept"] for r in new_rows}

for _, row in figures_new.iterrows():
    n = row["norm"]
    if n not in existing_new_norms and len(n) >= 2:
        new_rows.append({
            "concept": n,
            "display_name": row["label"],
            "entity_type": "PERSON",
            "notable_signal": row["sitelinks"],
            "source_origin": "wikidata_figures",
            "qid": row["qid"],
        })
        existing_new_norms.add(n)

# New places (not already in Jeopardy)
places_new = places[~places["already_in_jeopardy"]].copy()
places_new = places_new[~places_new["norm"].isin(all_existing)]

for _, row in places_new.iterrows():
    n = row["norm"]
    if n not in existing_new_norms and len(n) >= 2:
        new_rows.append({
            "concept": n,
            "display_name": row["label"],
            "entity_type": row["entity_type"],
            "geo_signal": row["sitelinks"],
            "source_origin": "wikidata_places",
            "qid": row["qid"],
        })
        existing_new_norms.add(n)

new_df = pd.DataFrame(new_rows)
print(f"New concepts to append: {len(new_df):,}")
print(f"\nBy source_origin:")
print(new_df["source_origin"].value_counts().to_string())
print(f"\nBy entity_type:")
print(new_df["entity_type"].value_counts().to_string())

In [ ]:
# Tag existing concepts with source_origin
composite_v1["source_origin"] = "jeopardy"

# Concatenate existing + new
composite_v2 = pd.concat([composite_v1, new_df], ignore_index=True)

# Fill NaN norm for new rows
composite_v2["norm"] = composite_v2["norm"].fillna(composite_v2["concept"].apply(normalize))

print(f"Composite v2 total: {len(composite_v2):,}")
print(f"  From existing: {len(composite_v1):,}")
print(f"  New additions: {len(new_df):,}")

## 6. Compute Expanded Percentile Ranks

In [ ]:
# Recompute existing percentile columns (now on the larger dataset)
composite_v2["pctile_jeopardy"] = composite_v2["frequency"].rank(pct=True) * 100
# Only rank where we have actual data
composite_v2.loc[composite_v2["frequency"].isna(), "pctile_jeopardy"] = np.nan

composite_v2["pctile_wiki"] = composite_v2["wiki_views"].rank(pct=True) * 100

composite_v2["pctile_xw"] = composite_v2["xw_frequency"].replace(0, np.nan).rank(pct=True) * 100

# New percentile columns
composite_v2["pctile_proverb"] = composite_v2["proverb_signal"].rank(pct=True) * 100

composite_v2["pctile_notable"] = composite_v2["notable_signal"].rank(pct=True) * 100

composite_v2["pctile_geo"] = composite_v2["geo_signal"].rank(pct=True) * 100

# Count signals and compute composite
pctile_cols = ["pctile_jeopardy", "pctile_wiki", "pctile_xw",
               "pctile_proverb", "pctile_notable", "pctile_geo"]

composite_v2["n_signals"] = composite_v2[pctile_cols].notna().sum(axis=1)
composite_v2["composite_score"] = composite_v2[pctile_cols].mean(axis=1)

print(f"Signal coverage:")
print(composite_v2["n_signals"].value_counts().sort_index().to_string())
print(f"\nPercentile coverage per signal:")
for col in pctile_cols:
    n = composite_v2[col].notna().sum()
    print(f"  {col:20s} {n:>7,} ({100*n/len(composite_v2):.1f}%)")

## 7. Top Concepts by Expanded Composite

In [ ]:
# Top 50 overall (require n_signals >= 2)
multi_signal = composite_v2[composite_v2["n_signals"] >= 2].copy()
print(f"Concepts with n_signals >= 2: {len(multi_signal):,}")

top50 = multi_signal.nlargest(50, "composite_score")
print(f"\nTop 50 by expanded composite score:")
display_cols = ["concept", "display_name", "entity_type", "n_signals", "composite_score"]
display_cols += [c for c in pctile_cols if c in top50.columns]
print(top50[display_cols].to_string(index=False, float_format="{:.1f}".format))

In [ ]:
# Top new concepts (from supplementary sources only)
new_concepts = composite_v2[composite_v2["source_origin"] != "jeopardy"].copy()
new_with_signal = new_concepts[new_concepts["n_signals"] >= 1].nlargest(30, "composite_score")

print(f"New concepts from supplementary sources: {len(new_concepts):,}")
print(f"New concepts with >= 1 signal: {new_concepts[new_concepts['n_signals'] >= 1].shape[0]:,}")
print(f"\nTop 30 new concepts by composite score:")
print(new_with_signal[["concept", "display_name", "entity_type", "source_origin",
                        "n_signals", "composite_score"]].to_string(index=False))

## 8. Re-run Hirsch Comparison

In [ ]:
# Load Hirsch terms
hirsch = pd.read_csv(f"{DATA_DIR}/hirsch_cultural_literacy_terms.csv")

# Normalize Hirsch terms
hirsch["norm"] = hirsch["term"].apply(normalize)

def flip_name(term):
    if ", " in str(term):
        parts = term.split(", ", 1)
        if len(parts) == 2 and parts[1][0:1].isupper():
            return normalize(parts[1] + " " + parts[0])
    return None

hirsch["norm_flipped"] = hirsch["term"].apply(flip_name)
hirsch_norms = set(hirsch["norm"].dropna())
hirsch_flipped = set(hirsch["norm_flipped"].dropna())
hirsch_all = hirsch_norms | hirsch_flipped

# Check coverage in expanded composite
v2_norms = set(composite_v2["norm"].dropna())
v2_display_norms = set(composite_v2["norm_display"].dropna()) if "norm_display" in composite_v2.columns else set()
# Also normalize display_name for new concepts
composite_v2["norm_display"] = composite_v2["display_name"].apply(normalize)
v2_display_norms = set(composite_v2["norm_display"].dropna())
v2_all = v2_norms | v2_display_norms

hirsch["in_v1"] = hirsch["norm"].isin(all_existing) | hirsch["norm_flipped"].isin(all_existing)
hirsch["in_v2"] = hirsch["norm"].isin(v2_all) | hirsch["norm_flipped"].isin(v2_all)

v1_coverage = hirsch["in_v1"].sum()
v2_coverage = hirsch["in_v2"].sum()

print(f"Hirsch Dictionary Coverage:")
print(f"  v1 (Jeopardy-anchored): {v1_coverage:,} / {len(hirsch):,} ({100*v1_coverage/len(hirsch):.1f}%)")
print(f"  v2 (expanded):          {v2_coverage:,} / {len(hirsch):,} ({100*v2_coverage/len(hirsch):.1f}%)")
print(f"  Improvement: +{v2_coverage - v1_coverage:,} terms ({100*(v2_coverage-v1_coverage)/len(hirsch):.1f}pp)")

In [ ]:
# Hirsch coverage by section: v1 vs v2
section_cov = hirsch.groupby("section").agg(
    total=("term", "size"),
    v1_matched=("in_v1", "sum"),
    v2_matched=("in_v2", "sum"),
).reset_index()
section_cov["v1_pct"] = 100 * section_cov["v1_matched"] / section_cov["total"]
section_cov["v2_pct"] = 100 * section_cov["v2_matched"] / section_cov["total"]
section_cov["improvement"] = section_cov["v2_pct"] - section_cov["v1_pct"]
section_cov = section_cov.sort_values("improvement", ascending=False)

print("Hirsch coverage by section (v1 -> v2):")
print(section_cov.to_string(index=False, float_format="{:.1f}".format))

# Plot comparison
fig, ax = plt.subplots(figsize=(14, 8))
sc_sorted = section_cov.sort_values("v2_pct")
y = np.arange(len(sc_sorted))
ax.barh(y - 0.2, sc_sorted["v1_pct"], 0.4, label="v1 (Jeopardy-anchored)", alpha=0.7)
ax.barh(y + 0.2, sc_sorted["v2_pct"], 0.4, label="v2 (expanded)", alpha=0.7)
ax.set_yticks(y)
ax.set_yticklabels(sc_sorted["section"])
ax.set_xlabel("% of Hirsch entries covered")
ax.set_title("Hirsch Coverage by Section: v1 vs v2")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Which Hirsch terms were newly covered in v2?
newly_covered = hirsch[(~hirsch["in_v1"]) & (hirsch["in_v2"])].copy()
print(f"Hirsch terms newly covered in v2: {len(newly_covered):,}")
print(f"\nBy section:")
print(newly_covered["section"].value_counts().head(15).to_string())
print(f"\nSample newly covered terms:")
for section in newly_covered["section"].value_counts().head(5).index:
    subset = newly_covered[newly_covered["section"] == section]
    print(f"\n  {section}:")
    for _, row in subset.head(8).iterrows():
        print(f"    {row['term']}")

In [ ]:
# Still-missing Hirsch terms
still_missing = hirsch[~hirsch["in_v2"]].copy()
print(f"Hirsch terms still missing from v2: {len(still_missing):,}")
print(f"\nBy section:")
print(still_missing["section"].value_counts().head(10).to_string())
print(f"\nSample still-missing terms:")
for section in still_missing["section"].value_counts().head(5).index:
    subset = still_missing[still_missing["section"] == section]
    print(f"\n  {section}:")
    for _, row in subset.head(8).iterrows():
        print(f"    {row['term']}")

## 8b. Fuzzy Matching to Recover Hirsch Near-Misses

Two high-precision fuzzy matching layers to recover terms missed by exact normalization:
- **Layer 1**: Unicode/diacritics normalization (strips combining characters, normalizes quotes)
- **Layer 2**: Levenshtein edit distance for short terms (catches OCR artifacts like missing ligatures)

In [ ]:
import unicodedata
from rapidfuzz import fuzz, process

# Enhanced normalization that handles Unicode issues
def normalize_fuzzy(text):
    """Aggressive normalization: strip diacritics, normalize quotes, lowercase."""
    if not isinstance(text, str):
        return ""
    # Normalize Unicode (NFKD decomposes diacritics)
    s = unicodedata.normalize("NFKD", text)
    # Remove combining characters (diacritics)
    s = "".join(c for c in s if not unicodedata.combining(c))
    # Normalize all quote-like characters to nothing
    s = re.sub(r"[\u2018\u2019\u201C\u201D\u0060\u00B4'\"`]", "", s)
    # Standard normalization
    s = s.lower().strip()
    s = re.sub(r"^(the|a|an)\s+", "", s)
    s = re.sub(r"[,;:!?._()\[\]{}<>*/\\@#$%^&+=~|]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    # Remove trailing hyphens/dashes
    s = s.rstrip("- ")
    return s

# Detect truncated Hirsch entries (ending mid-word)
def is_truncated(term):
    """Check if a Hirsch term appears to be truncated."""
    if not isinstance(term, str):
        return False
    stripped = term.rstrip()
    # Ends without punctuation and last word is short (likely cut off)
    if stripped and stripped[-1].isalpha():
        words = stripped.split()
        if len(words) > 2 and len(words[-1]) <= 3:
            return True
        # Check if it just ends abruptly (no period, question mark, etc.)
        if len(stripped) > 30:
            return True
    return False

# Build the lookup sets with fuzzy normalization
still_missing = hirsch[~hirsch["in_v2"]].copy()
still_missing["norm_fuzzy"] = still_missing["term"].apply(normalize_fuzzy)
still_missing["norm_flipped_fuzzy"] = still_missing["term"].apply(
    lambda t: normalize_fuzzy(flip_name(t)) if flip_name(t) else None
)
still_missing["truncated"] = still_missing["term"].apply(is_truncated)

print(f"Still missing before fuzzy matching: {len(still_missing):,}")
print(f"  Truncated entries: {still_missing['truncated'].sum():,}")
print(f"\nSample truncated entries:")
for _, row in still_missing[still_missing["truncated"]].head(10).iterrows():
    print(f"  |{row['term']}|")

In [ ]:
# Build fuzzy-normalized lookup from composite v2
composite_v2["norm_fuzzy"] = composite_v2["concept"].apply(normalize_fuzzy)
composite_v2["norm_display_fuzzy"] = composite_v2["display_name"].apply(normalize_fuzzy)

v2_fuzzy_norms = set(composite_v2["norm_fuzzy"].dropna()) | set(composite_v2["norm_display_fuzzy"].dropna())
v2_fuzzy_norms.discard("")

print(f"Composite v2 fuzzy-normalized unique strings: {len(v2_fuzzy_norms):,}")

# ===== LAYER 1: Re-match with improved Unicode normalization =====
layer1_matches = []
for idx, row in still_missing.iterrows():
    nf = row["norm_fuzzy"]
    nff = row["norm_flipped_fuzzy"]
    if nf in v2_fuzzy_norms or (nff and nff in v2_fuzzy_norms):
        layer1_matches.append(idx)

print(f"\nLayer 1 (Unicode/diacritics fix): {len(layer1_matches):,} new matches")
if layer1_matches:
    sample = still_missing.loc[layer1_matches[:15], "term"].tolist()
    print(f"  Samples: {sample}")

In [ ]:
# ===== LAYER 2: Character-level edit distance for short terms =====
# Levenshtein ratio catches OCR artifacts (missing ligatures like fl/fi),
# minor spelling differences, and hyphenation variants

already_matched = set(layer1_matches)
remaining = still_missing[~still_missing.index.isin(already_matched)].copy()
short_remaining = remaining[remaining["norm_fuzzy"].str.len() < 30].copy()
print(f"Remaining after Layer 1: {len(remaining):,}")
print(f"Short terms (< 30 chars) for edit distance: {len(short_remaining):,}")

# Build choice list from composite fuzzy norms
choices = list(v2_fuzzy_norms)

THRESHOLD_LEVENSHTEIN = 88

layer2_matches = []
layer2_details = []

for idx, row in short_remaining.iterrows():
    nf = row["norm_fuzzy"]
    nff = row["norm_flipped_fuzzy"]
    if not isinstance(nff, str):
        nff = None
    if not nf or len(nf) < 3:
        continue
    
    best_score = 0
    best_match = ""
    
    queries = [nf] + ([nff] if nff else [])
    for query in queries:
        result = process.extractOne(
            query, choices,
            scorer=fuzz.ratio,
            score_cutoff=THRESHOLD_LEVENSHTEIN,
        )
        if result and result[1] > best_score:
            best_score = result[1]
            best_match = result[0]
    
    if best_score >= THRESHOLD_LEVENSHTEIN:
        layer2_matches.append(idx)
        layer2_details.append((nf, best_match, best_score))

print(f"\nLayer 2 (Levenshtein ratio >= {THRESHOLD_LEVENSHTEIN}): {len(layer2_matches):,} new matches")
print(f"\nSample matches:")
sorted_l2 = sorted(zip(layer2_matches, layer2_details), key=lambda x: -x[1][2])
for idx, (query, match, score) in sorted_l2[:20]:
    print(f"  [{score:3.0f}] {still_missing.loc[idx, 'term']:40s} -> {match}")

In [ ]:
# ===== Consolidate fuzzy matches and update Hirsch coverage =====
all_fuzzy_matched = set(layer1_matches) | set(layer2_matches)

print(f"Fuzzy matching summary:")
print(f"  Layer 1 (Unicode/diacritics):      {len(layer1_matches):>5,}")
print(f"  Layer 2 (Levenshtein ratio >= 88): {len(layer2_matches):>5,}")
print(f"  -----------------------------------------")
print(f"  Total new fuzzy matches:           {len(all_fuzzy_matched):>5,}")

# Update Hirsch coverage
hirsch["in_v2_fuzzy"] = hirsch["in_v2"] | hirsch.index.isin(all_fuzzy_matched)
v2_fuzzy_coverage = hirsch["in_v2_fuzzy"].sum()

print(f"\nHirsch Dictionary Coverage (updated):")
print(f"  v1 (Jeopardy-anchored):     {v1_coverage:,} / {len(hirsch):,} ({100*v1_coverage/len(hirsch):.1f}%)")
print(f"  v2 (exact match):           {v2_coverage:,} / {len(hirsch):,} ({100*v2_coverage/len(hirsch):.1f}%)")
print(f"  v2 + fuzzy:                 {v2_fuzzy_coverage:,} / {len(hirsch):,} ({100*v2_fuzzy_coverage/len(hirsch):.1f}%)")
print(f"  Total improvement over v1:  +{v2_fuzzy_coverage - v1_coverage:,} terms ({100*(v2_fuzzy_coverage-v1_coverage)/len(hirsch):.1f}pp)")

In [ ]:
# Coverage by section with fuzzy matching
section_cov_fuzzy = hirsch.groupby("section").agg(
    total=("term", "size"),
    v1_matched=("in_v1", "sum"),
    v2_exact=("in_v2", "sum"),
    v2_fuzzy=("in_v2_fuzzy", "sum"),
).reset_index()
section_cov_fuzzy["v1_pct"] = 100 * section_cov_fuzzy["v1_matched"] / section_cov_fuzzy["total"]
section_cov_fuzzy["v2_exact_pct"] = 100 * section_cov_fuzzy["v2_exact"] / section_cov_fuzzy["total"]
section_cov_fuzzy["v2_fuzzy_pct"] = 100 * section_cov_fuzzy["v2_fuzzy"] / section_cov_fuzzy["total"]
section_cov_fuzzy["fuzzy_gain"] = section_cov_fuzzy["v2_fuzzy_pct"] - section_cov_fuzzy["v2_exact_pct"]
section_cov_fuzzy = section_cov_fuzzy.sort_values("v2_fuzzy_pct", ascending=False)

print("Hirsch coverage by section (v1 -> v2 exact -> v2 fuzzy):")
print(section_cov_fuzzy[["section", "total", "v1_pct", "v2_exact_pct", "v2_fuzzy_pct", "fuzzy_gain"]].to_string(
    index=False, float_format="{:.1f}".format))

In [ ]:
# What's still missing after all fuzzy matching?
final_missing = hirsch[~hirsch["in_v2_fuzzy"]].copy()
print(f"Still missing after all matching: {len(final_missing):,} / {len(hirsch):,} ({100*len(final_missing)/len(hirsch):.1f}%)")
print(f"\nBy section:")
print(final_missing["section"].value_counts().to_string())
print(f"\nSample still-missing terms by top sections:")
for section in final_missing["section"].value_counts().head(6).index:
    subset = final_missing[final_missing["section"] == section]
    print(f"\n  {section} ({len(subset):,} missing):")
    for _, row in subset.head(10).iterrows():
        print(f"    {row['term']}")

## 9. Spot Checks

In [ ]:
# Spot-check specific concepts from the plan
spot_checks = [
    "absence makes the heart grow fonder",
    "confucius",
    "cervantes",
    "goethe",
    "marthas vineyard",
    "appalachia",
]

print("Spot checks:")
for check in spot_checks:
    norm_check = normalize(check)
    matches = composite_v2[
        (composite_v2["norm"] == norm_check) |
        (composite_v2["norm_display"] == norm_check) |
        (composite_v2["norm"].str.contains(norm_check, na=False))
    ]
    if len(matches) > 0:
        row = matches.iloc[0]
        signals = []
        for col in pctile_cols:
            if pd.notna(row.get(col)):
                signals.append(f"{col}={row[col]:.1f}")
        print(f"  [FOUND] {check} -> {row.get('display_name', row['concept'])}")
        print(f"          entity_type={row['entity_type']}, n_signals={row.get('n_signals', 0)}, " +
              f"composite={row.get('composite_score', 0):.1f}")
        print(f"          signals: {', '.join(signals)}")
        print(f"          source: {row.get('source_origin', 'jeopardy')}")
    else:
        print(f"  [MISSING] {check}")

## 10. Export

In [ ]:
# Export v2 composite
export_cols = [
    "concept", "display_name", "entity_type",
    # Existing signals
    "frequency", "seasons_appeared", "category_breadth", "importance",
    "wiki_title", "wiki_views", "wiki_monthly_avg",
    "xw_frequency",
    # New signals
    "proverb_signal", "notable_signal", "geo_signal",
    # All percentiles
    "pctile_jeopardy", "pctile_wiki", "pctile_xw",
    "pctile_proverb", "pctile_notable", "pctile_geo",
    # Composite
    "n_signals", "composite_score",
    # Metadata
    "source_origin", "qid",
]

# Only include columns that exist
export_cols = [c for c in export_cols if c in composite_v2.columns]

export = composite_v2[export_cols].sort_values("composite_score", ascending=False)
output_path = f"{DATA_DIR}/cultural_salience_composite_v2.csv"
export.to_csv(output_path, index=False)
print(f"Exported {len(export):,} concepts to {output_path}")

In [ ]:
# Final summary
print("=" * 65)
print("EXPANDED COMPOSITE SUMMARY (v2)")
print("=" * 65)

print(f"\nDataset size:")
print(f"  v1 concepts: {len(composite_v1):,}")
print(f"  v2 concepts: {len(composite_v2):,} (+{len(composite_v2) - len(composite_v1):,})")

print(f"\nNew concepts by source:")
source_counts = composite_v2[composite_v2["source_origin"] != "jeopardy"]["source_origin"].value_counts()
for source, count in source_counts.items():
    print(f"  {source:25s} {count:>6,}")

print(f"\nSignal coverage (n_signals):")
print(composite_v2["n_signals"].value_counts().sort_index().to_string())

print(f"\nConcepts with n_signals >= 2: {(composite_v2['n_signals'] >= 2).sum():,}")
print(f"Concepts with n_signals >= 3: {(composite_v2['n_signals'] >= 3).sum():,}")

print(f"\nHirsch coverage:")
print(f"  v1: {v1_coverage:,} / {len(hirsch):,} ({100*v1_coverage/len(hirsch):.1f}%)")
print(f"  v2: {v2_coverage:,} / {len(hirsch):,} ({100*v2_coverage/len(hirsch):.1f}%)")
print(f"  Improvement: +{v2_coverage - v1_coverage:,} terms")

print(f"\nEntity type breakdown (all v2):")
print(composite_v2["entity_type"].value_counts().head(15).to_string())

print(f"\nOutput: {output_path}")